In [17]:
import ctypes
import time
import numpy as np
import pandas as pd
from datetime import datetime

import csv, math, os, sys, copy, random, json
from numpy import asarray, savetxt
from scipy.optimize import curve_fit
from scipy.integrate import quad
from scipy.io import loadmat



from numba import njit
from line_profiler import LineProfiler

####### image single tweezer with camera
from PIL import Image
import matplotlib.pyplot as plt
import cv2
import imageio

import labrad
import sys
sys.path.append("C://Users/Cryo_rdyberg/Documents/Codebase/Lab_control/experiment_scripts_Cs/experiment_phases")
sys.path.append("C://Users/Cryo_rdyberg/Documents/Codebase/Lab_control/servers/script_scanner/")



from labrad.units import WithUnit
from json import JSONEncoder


# %load_ext autoreload
# %autoreload 2  # automatically reloads changes made to labrad_helpers.py
import sys, os
def add_path_up(levels=2):
    target = os.path.abspath(os.path.join(os.getcwd(), *['..']*levels))
    if target not in sys.path:
        sys.path.append(target)
    print("Added to path:", target)


add_path_up(2)

#IMPORT PHASES
from Experiment_config import *
from experiment_phases import experiment_phases, S, P
#IMPORT SCAN EXPERIMENT BASE CLASSES
from single_sequence import single_sequence
from scan_experiment import scan_experiment
from imaging import imaging
from load_MOT_Tweezers import load_MOT_Tweezers
from shortPGC import shortPGC
from adiabatic_cooling import adiabatic_cooling
from optical_pumping import optical_pumping
from single_pi_pulse import single_pi_pulse
from fast_pushout import fast_pushout
from recap import recap
from thermalize import thermalize
from set_final_state import set_final_state

Added to path: c:\Users\Cryo_rdyberg\Documents\Codebase\Lab_control


In [18]:
exp_name = "test_control_imaging"
def exp_sequence():
    return S(load_MOT_Tweezers(),imaging(),shortPGC(),imaging(),thermalize(),set_final_state())

Scan_points = 4
scan_var_start=WithUnit(1e-6,'s')
scan_var_end=WithUnit(2e-6,'s')
parameter= [("general", "scan_variable")]
#parameter= [("single_pi_pulse", "uv_duration_sp")]



In [19]:
#### reload exp phases, need to rerun for reloading after change (does not require restarting kernel)
#IMPORT PHASES
from Experiment_config import *
from experiment_phases import experiment_phases, S, P
#IMPORT SCAN EXPERIMENT BASE CLASSES
from single_sequence import single_sequence
from scan_experiment import scan_experiment
from imaging import imaging
from load_MOT_Tweezers import load_MOT_Tweezers
from shortPGC import shortPGC
from adiabatic_cooling import adiabatic_cooling
from optical_pumping import optical_pumping
from single_pi_pulse import single_pi_pulse
from fast_pushout import fast_pushout
from recap import recap
from thermalize import thermalize
from set_final_state import set_final_state


# def reload_config():
#     import importlib, Experiment_config_v18_7_1219 as Experiment_config_new_seq
#     importlib.reload(Experiment_config_new_seq)
#     globals().update(vars(Experiment_config_new_seq))

# reload_config()






from datetime import date
today = date.today().strftime("\%Y\%m\%d")
# Full path to target folder
target_path = os.path.join(folder_path+today+"\\", exp_name)

# Check and create if needed
if not os.path.exists(target_path+"\\pvcam"):
    os.makedirs(target_path+"\\pvcam")
    print(f"Created folder: {target_path}")

cxn = labrad.connect()
#make saving dirc
cxn.parametervault.set_parameter("general","target_path",target_path)
#save scan para name
cxn.parametervault.set_parameter("general","scan_variable_name",parameter[0][1])
cxn.parametervault.set_parameter("general","scan_variable",(scan_var_start["s"],scan_var_end["s"],Scan_points,"s"))

cxn.parametervault.get_parameter("general","target_path")


Created folder: C:\Users\Cryo_rdyberg\Princeton Dropbox\Yukai Lu\CryoRydberg\Data\2026\01\21\test_control_imaging


'C:\\Users\\Cryo_rdyberg\\Princeton Dropbox\\Yukai Lu\\CryoRydberg\\Data\\2026\\01\\21\\test_control_imaging'

In [20]:

#cxn.parametervault.set_parameter("load_MOT_Tweezers","bias_zv_for_loading",WithUnit(1.0,"V"))
cxn.parametervault.reload_parameters()
cxn.parametervault.get_parameter("load_MOT_Tweezers","bias_zv_for_loading")

Value(1.0, 'V')

In [21]:

class run_exp_(experiment_phases):
    #for all parameters that are used in the experiment but not in the phases included
    required_parameters = [("general","verbose"),("general","seqlen"),("general","nLoops"),("general","target_path"),("general","scan_variable_name"),("general","scan_variable")]
    name = "run_exp_"
    first = None
    phases = None
    last = None

    def __init__(self, name=None, required_parameters=None, cxn=None, min_progress=0.0, max_progress=100.0):
        #self.phases, self.first, self.last,load_required_parameters = exp_sequence()
        #self.required_parameters += load_required_parameters 

        #need to be removed later, depending if we need global variables like an array (or we can just pass a dir in parameter vault)
        #now self.params["seqlen"] are used for calculating time for a single run
        self.params = {}
        self.results = []
        super().__init__(self.name, self.required_parameters, cxn,min_progress, max_progress)

    @classmethod
    def all_required_parameters(cls):
        #print("running it")
        if cls.first == None:
            cls.required_parameters += cls._load_parameters_before_init()
        parameters = set(cls.required_parameters)
        parameters = list(parameters)
        #print(cls.required_parameters)
        #print("test")
        return parameters
    
    #This function is used for loading parameters for the SC GUI. In the SC GUI it will call @classmethod all_required_parameters to load paramters
    #@classmethod function will run itself before the class is initialized, that's why we need this function here to avoid an empty parameters
    # we also used it to initialize our experiment seq by letting cls.phase, cls.first cls.last to be what needed to be
    #In this way we do not need to rerun exp_sequence again to find out the first and last phases
    @classmethod
    def _load_parameters_before_init(cls):
        cls.phases, cls.first, cls.last,required_parameters = exp_sequence()
        #print(cls.first)
        return required_parameters



scanner = cxn.scriptscanner
###: you set the scan freqeuncy and data points needed here

#exprt = scan_experiment(run_exp_, parameter, scan_time_start['s'], scan_time_end['s'], Scan_points, 's')
assert(scan_var_start.units==scan_var_end.units)
exprt = scan_experiment(run_exp_, parameter, scan_var_start[scan_var_start.units], scan_var_end[scan_var_end.units], Scan_points, scan_var_start.units)
ident = scanner.register_external_launch(exprt.name)
exprt.execute(ident)

init, set up pulser
('general', 'scan_variable') changed to 1e-06 s
init, set up pulser
before do phase
D
1.705308437347412
run waveforms, number of loops: 20.0
scantime: 23.000000000000004
F
25.443434715270996
('general', 'scan_variable') changed to 1.3333333333333332e-06 s
init, set up pulser
before do phase
D
1.634298324584961
run waveforms, number of loops: 20.0
scantime: 23.000000000000004
F
25.37433433532715
('general', 'scan_variable') changed to 1.6666666666666667e-06 s
init, set up pulser
before do phase
D
1.7393462657928467
run waveforms, number of loops: 20.0
scantime: 23.000000000000004
F
25.40520405769348
('general', 'scan_variable') changed to 2e-06 s
init, set up pulser
before do phase
wait 1.0s for acquistion to get ready
D
2.487478733062744
run waveforms, number of loops: 20.0
scantime: 23.000000000000004
F
26.248633861541748
finalize


In [5]:
type(target_path)

str

In [ ]:

    




cxn = labrad.connect()
scanner = cxn.scriptscanner
###: you set the scan freqeuncy and data points needed here
exprt = single_sequence(run_exp_)
ident = scanner.register_external_launch(exprt.name)
exprt.execute(ident)

init, set up pulser
finalize
